In [ ]:
import polars as pl
from datetime import date
df_amostra = pl.scan_parquet(r'amostra_noticias_v1.parquet')
df_noticias = df_amostra.with_columns(
    pl.col("data_iso")
    .str.to_date("%Y-%m-%d")
    .alias("data_iso")
)




In [ ]:
df_noticias.head(5).collect()

## Análises dos dados

In [ ]:
import polars as pl
import matplotlib.pyplot as plt


df_noticias = df_noticias.with_columns(
        pl.col("data_iso")
        .dt.strftime("%Y-%m")
        .alias("ano_mes")
    )


noticias_por_mes = (
    df_noticias
    .filter(pl.col("data_iso").is_not_null())
    .group_by("ano_mes")
    .len(name="quantidade")
    .sort("ano_mes")
)

noticias_por_mes = noticias_por_mes.collect()


## Volumetria mensal

In [ ]:

plt.figure(figsize=(14, 5))

plt.bar(
    noticias_por_mes["ano_mes"].to_list(),
    noticias_por_mes["quantidade"].to_list()
)

plt.xlabel("Mês")
plt.ylabel("Número de notícias")
plt.title("Volumetria mensal de notícias")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Caracterizando autores e tentando encontrar alguma analise

In [ ]:
dim_autores = (
    df_noticias
    .select("autor")
    .filter(pl.col("autor").is_not_null())
    .unique()
    .sort("autor")
    .with_row_index(name="id_autor", offset=1)
)

dim_autores.select('autor').count().collect()

In [ ]:
df_noticias_com_id = (
    df_noticias
    .join(dim_autores, on="autor", how="left")
)


In [ ]:
analise_autores = (
    df_noticias_com_id
    .group_by(["id_autor",'autor'])
    .agg([
        pl.len().alias("qtd_noticias"),
        pl.col("data_iso").min().alias("primeira_publicacao"),
        pl.col("data_iso").max().alias("ultima_publicacao"),
        pl.col("ano_mes").n_unique().alias("qtd_meses_com_publicacao"),
        pl.col("texto").str.len_chars().mean().alias("tamanho_medio_texto")
    ])
    .with_columns([
        (
            pl.col("ultima_publicacao") - pl.col("primeira_publicacao")
        ).dt.total_days().alias("dias_entre_primeira_e_ultima")
    ])
    .sort("qtd_noticias", descending=True)
)


analise_autores.head(10).collect().to_pandas()
## Existem autores que diminuem o relacionamento conforme passa o tempo, talvez seja de fato possivel modelar demissao aqui

# Criando foto

In [ ]:
autores = (
    df_noticias_com_id
    .select('autor','id_autor')
    .filter(
        pl.col('autor').is_not_null()
    )
    .unique()
)

In [ ]:
datas_ref = pl.LazyFrame({
    "data_ref": pl.date_range(
        start=date(2022, 6, 1),
        end=date(2026, 5, 1),
        interval="1mo",
        eager=True
    )
})

datas_ref.head(10).collect()

In [ ]:
matriz_mod = (
    autores
    .join(datas_ref, how="cross")
    .sort(["id_autor", "data_ref"])
)

matriz_mod.head(5).collect()

In [ ]:
primeira_publi = (
    df_noticias_com_id
    .filter(
        pl.col('id_autor').is_not_null()
    )
    .group_by('id_autor')
    .agg(
        pl.col('data_iso').min().alias('menor_data_publicacao'),
        pl.col('data_iso').max().alias('maior_data_publicacao'),
    )
)

matriz_mod = (
    matriz_mod
    .join(
        primeira_publi, on = 'id_autor', how = 'left'
    )
    .filter(  (pl.col('data_ref') >= pl.col('menor_data_publicacao')) 
            & (pl.col('data_ref') <= pl.col('maior_data_publicacao')))
    .with_columns(
        pl.when(
            pl.col('menor_data_publicacao') <= pl.col('data_ref').dt.offset_by('-1mo')
        )
        .then(1)
        .otherwise(0)
        .alias('flag_1m_relacionamento'),


        pl.when(
            pl.col('menor_data_publicacao') <= pl.col('data_ref').dt.offset_by('-3mo')
        )
        .then(1)
        .otherwise(0)
        .alias('flag_3m_relacionamento'),


        pl.when(
            pl.col('menor_data_publicacao') <= pl.col('data_ref').dt.offset_by('-6mo')
        )
        .then(1)
        .otherwise(0)
        .alias('flag_6m_relacionamento'),
    )
    .with_columns(
        pl.when(
            pl.col('flag_3m_relacionamento')==0
        )
        .then(pl.lit('EM_MATURACAO'))
        .otherwise(pl.lit('MATURADO'))
        .alias('SEGMENTO'),
        
        pl.col('data_ref').dt.offset_by('3mo').alias('data_fim_janela_3m'),
        pl.lit(date(2026, 5, 1)).alias("elegib_temporal_conceito")
    )
)

matriz_mod.head(3).collect()

In [ ]:
lf_matriz = matriz_mod
lf_noticias = df_noticias_com_id

qtd_noticias_antes_ref = (
    lf_matriz
    .select(["id_autor", "data_ref"])
    .unique()
    .join(
        lf_noticias
        .select(["id_autor", "data_iso"])
        .filter(
            pl.col("id_autor").is_not_null() &
            pl.col("data_iso").is_not_null()
        ),
        on="id_autor",
        how="left"
    )
    .filter(
        pl.col("data_iso") < pl.col("data_ref")
    )
    .group_by(["id_autor", "data_ref"])
    .agg(
        pl.len().alias("qtd_noticias_antes_ref")
    )
)

matriz_mod = (
    lf_matriz
    .join(
        qtd_noticias_antes_ref,
        on=["id_autor", "data_ref"],
        how="left"
    )
    .with_columns(
        pl.col("qtd_noticias_antes_ref")
        .fill_null(0)
    )
)

In [ ]:
(
    matriz_mod
    .with_columns(
        pl.when(
            (pl.col("qtd_noticias_antes_ref") >= 30) &
            (pl.col("flag_6m_relacionamento") == 1) 
            )
        .then(1)
        .otherwise(0)
        .alias("flag_pelo_menos_15_noticias_antes_ref")
    )
    .group_by("flag_pelo_menos_15_noticias_antes_ref")
    .agg(
        pl.len().alias("volume")
    )

).head(10).collect()

# Testando novo conceito de segmentacao

In [ ]:
matriz_mod = (
    matriz_mod
    .with_columns(
        pl.when(
            (pl.col("flag_6m_relacionamento") == 1) &
            (pl.col("qtd_noticias_antes_ref") >= 30)
        )
        .then(pl.lit("MATURADO"))
        .otherwise(pl.lit("EM_MATURACAO"))
        .alias("SEGMENTO")
    )
)

# Variavel resposta conceito de demissao

In [ ]:
qtd_publicacoes_futuras_3m = (
    matriz_mod
    .select('id_autor','data_ref','data_fim_janela_3m')
    .join(
        df_noticias_com_id,
        on= 'id_autor',
        how = 'left'
    )
    .filter(
        (pl.col('data_iso') >= pl.col('data_ref')) &
        (pl.col('data_iso') < pl.col('data_fim_janela_3m'))
    )
    .group_by('id_autor','data_ref')
    .agg(
        pl.len().alias('qtd_publicacoes_futuras_3m')
    )
)




In [ ]:
matriz_mod = (
    matriz_mod
    .join(
        qtd_publicacoes_futuras_3m,
        on = ['id_autor','data_ref'],
        how='left'
    )
    .with_columns(
        pl.col('qtd_publicacoes_futuras_3m').fill_null(0)
    )
    .with_columns(
        pl.when(
            pl.col('qtd_publicacoes_futuras_3m')<15
        )
        .then(1)
        .otherwise(0)
        .alias('conceito_demissao_3m')
    )
    .with_columns(
        pl.when(
            pl.col('data_fim_janela_3m') <= pl.col('elegib_temporal_conceito')
        )
        .then(pl.col('conceito_demissao_3m'))
        .otherwise(None)
        .alias('conceito_demissao_3m_real')
    )
    .filter(pl.col('data_ref')> pl.date(2022,12,1))
    .with_columns(
        pl.when(
            pl.col('data_ref') == pl.date(2026,2,1)
        )
        .then(None)
        .otherwise(pl.col('conceito_demissao_3m_real'))
        .alias('conceito_demissao_3m_real')
    )
)

In [ ]:
matriz_mod.sink_parquet(r'temp_matriz_dados.parquet')
matriz_mod = pl.scan_parquet(r'temp_matriz_dados.parquet')

In [ ]:
matriz_mod.head(5).collect()

In [ ]:
res_grafico = (
    matriz_mod
    .filter(
        (pl.col("SEGMENTO") == "MATURADO") 
    )
    .group_by("data_ref")
    .agg([
        pl.len().alias("volume_autores"),


        pl.col("conceito_demissao_3m_real").sum().alias("qtd_demissao_3m"),
        pl.col("conceito_demissao_3m_real").mean().alias("taxa_demissao_3m"),


    ])
    .with_columns(
        (pl.col("taxa_demissao_3m") * 100).alias("taxa_demissao_3m_pct"),

    )
    .sort("data_ref")
)

res_grafico = (
    res_grafico.collect())

res_grafico.head(55).to_pandas()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.bar(
    res_grafico["data_ref"].to_list(),
    res_grafico["volume_autores"].to_list(),
    width=20,
    alpha=0.7,
    label="Volume de autores maturados"
)

ax1.set_xlabel("Data de referência")
ax1.set_ylabel("Volume de autores maturados")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()

ax2.plot(
    res_grafico["data_ref"].to_list(),
    res_grafico["taxa_demissao_3m_pct"].to_list(),
    marker="o",
    color="indianred",
    linewidth=2,
    label="Taxa de demissão 3m (%)"
)

ax2.set_ylabel("Taxa de demissão 3m (%)")

plt.title("Volume de autores maturados e taxa de demissão 3m por data de referência")

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Amostragem

In [ ]:
matriz_mod.head(5).collect()

In [ ]:
from datetime import date
import polars as pl

data_inicio_teste = date(2025, 9, 1)

base_modelagem = (
    matriz_mod
    .filter(
        pl.col("SEGMENTO") == "MATURADO"
    )
    .with_columns(
        pl.col("data_ref")
        .dt.strftime("%Y-%m")
        .alias("safra")
    )
    .with_columns(
        pl.when(
            pl.col("conceito_demissao_3m_real").is_null()
        )
        .then(pl.lit("TTD"))

        .when(
            pl.col("data_ref") < pl.lit(data_inicio_teste)
        )
        .then(pl.lit("DESENVOLVIMENTO"))

        .otherwise(pl.lit("TESTE"))
        .alias("AMOSTRA")
    )
)
base_modelagem = (
    base_modelagem
    .filter(
        pl.col('id_autor') != 121
    )
)

base_modelagem.head(3).collect()

In [ ]:
res_grafico = (
    base_modelagem
    .group_by(["data_ref", "AMOSTRA"])
    .agg([
        pl.len().alias("volume_autores"),
        pl.col("conceito_demissao_3m_real").mean().alias("taxa_demissao_3m")
    ])
    .with_columns(
        (pl.col("taxa_demissao_3m") * 100).alias("taxa_demissao_3m_pct")
    )
    .sort("data_ref")
)

res_grafico = res_grafico.collect()


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

cores_amostra = {
    "DESENVOLVIMENTO": "#4C72B0",  # azul suave
    "TESTE": "#55A868",            # verde suave
    "TTD": "#C7C7C7"               # cinza
}

cores_barras = [
    cores_amostra.get(amostra, "#999999")
    for amostra in res_grafico["AMOSTRA"].to_list()
]

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.bar(
    res_grafico["data_ref"].to_list(),
    res_grafico["volume_autores"].to_list(),
    width=20,
    alpha=0.75,
    color=cores_barras,
    label="Volume de autores maturados"
)

ax1.set_xlabel("Data de referência")
ax1.set_ylabel("Volume de autores maturados")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()

ax2.plot(
    res_grafico["data_ref"].to_list(),
    res_grafico["taxa_demissao_3m_pct"].to_list(),
    marker="o",
    color="indianred",
    linewidth=2,
    label="Taxa de demissão 3m (%)"
)

ax2.set_ylabel("Taxa de demissão 3m (%)")

plt.title("Volume, taxa de demissão 3m e amostra por data de referência")

# Legenda das barras por amostra
legenda_amostras = [
    mpatches.Patch(color=cor, label=amostra)
    for amostra, cor in cores_amostra.items()
]

# Legenda da linha
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles=legenda_amostras + handles2,
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
base_modelagem.columns

In [ ]:
base_modelagem = (
    base_modelagem.select(
    'autor',
 'id_autor',
 'data_ref',
 'menor_data_publicacao',
 'flag_1m_relacionamento',
 'flag_3m_relacionamento',
 'flag_6m_relacionamento',
 'SEGMENTO',
 'conceito_demissao_3m_real',
 'AMOSTRA'
    )
)
base_modelagem.sink_parquet(r'base_modelagem_v1.parquet')
base_modelagem = pl.scan_parquet(r'base_modelagem_v1.parquet')



In [ ]:
base_modelagem = pl.scan_parquet(r'base_modelagem_v1.parquet')
base_modelagem.head(10).collect()

# Criacao de variaveis

In [ ]:
def criar_covariaveis_janela(matriz_mod, df_noticias_com_id, janela_meses):

    sufixo = f"{janela_meses}m"

    base_ref = (
        matriz_mod
        .select(["id_autor", "data_ref"])
        .unique()
        .with_columns(
            pl.col("data_ref")
            .dt.offset_by(f"-{janela_meses}mo")
            .alias(f"data_inicio_janela_{sufixo}")
        )
    )

    noticias_hist = (
        df_noticias_com_id
        .select([
            "id_autor",
            "data_iso",
            "titulo",
            "texto",
            "pagina_origem"
        ])
        .filter(
            pl.col("id_autor").is_not_null() &
            pl.col("data_iso").is_not_null()
        )
        .with_columns([
            pl.col("data_iso")
            .dt.strftime("%Y-%m")
            .alias("ano_mes_publicacao"),

            pl.col("texto")
            .str.len_chars()
            .alias("tam_texto"),

            pl.col("titulo")
            .str.len_chars()
            .alias("tam_titulo")
        ])
    )

    covs = (
        base_ref
        .join(
            noticias_hist,
            on="id_autor",
            how="left"
        )
        .filter(
            (pl.col("data_iso") >= pl.col(f"data_inicio_janela_{sufixo}")) &
            (pl.col("data_iso") < pl.col("data_ref"))
        )
        .group_by(["id_autor", "data_ref"])
        .agg([
            pl.len().alias(f"qtd_publicacoes_{sufixo}"),

            pl.col("data_iso")
            .n_unique()
            .alias(f"qtd_dias_publicados_{sufixo}"),

            pl.col("ano_mes_publicacao")
            .n_unique()
            .alias(f"qtd_meses_com_publicacao_{sufixo}"),

            pl.col("data_iso")
            .max()
            .alias(f"ultima_publicacao_{sufixo}"),

            pl.col("tam_texto")
            .mean()
            .alias(f"tam_medio_texto_{sufixo}"),

            pl.col("tam_texto")
            .sum()
            .alias(f"tam_total_texto_{sufixo}"),

            pl.col("tam_titulo")
            .mean()
            .alias(f"tam_medio_titulo_{sufixo}"),

            pl.col("pagina_origem")
            .n_unique()
            .alias(f"qtd_paginas_origem_distintas_{sufixo}")
        ])
    )

    covs = (
        base_ref
        .select(["id_autor", "data_ref", f"data_inicio_janela_{sufixo}"])
        .join(
            covs,
            on=["id_autor", "data_ref"],
            how="left"
        )
        .with_columns([
            pl.col(f"qtd_publicacoes_{sufixo}")
            .fill_null(0),

            pl.col(f"qtd_dias_publicados_{sufixo}")
            .fill_null(0),

            pl.col(f"qtd_meses_com_publicacao_{sufixo}")
            .fill_null(0),

            pl.col(f"tam_total_texto_{sufixo}")
            .fill_null(0),

            pl.col(f"qtd_paginas_origem_distintas_{sufixo}")
            .fill_null(0)
        ])
        .with_columns([
            pl.when(pl.col(f"qtd_publicacoes_{sufixo}") > 0)
            .then(1)
            .otherwise(0)
            .alias(f"flag_publicou_{sufixo}"),

            (
                pl.col(f"qtd_publicacoes_{sufixo}") / janela_meses
            )
            .alias(f"media_publicacoes_mes_{sufixo}"),

            (
                pl.col(f"qtd_meses_com_publicacao_{sufixo}") / janela_meses
            )
            .alias(f"prop_meses_com_publicacao_{sufixo}"),

            (
                pl.col("data_ref") - pl.col(f"data_inicio_janela_{sufixo}")
            )
            .dt.total_days()
            .alias(f"qtd_dias_janela_{sufixo}")
        ])
        .with_columns([
            (
                pl.col(f"qtd_dias_publicados_{sufixo}") /
                pl.col(f"qtd_dias_janela_{sufixo}")
            )
            .alias(f"prop_dias_com_publicacao_{sufixo}"),

            pl.when(pl.col(f"qtd_publicacoes_{sufixo}") > 0)
            .then(
                (pl.col("data_ref") - pl.col(f"ultima_publicacao_{sufixo}"))
                .dt.total_days()
            )
            .otherwise(None)
            .alias(f"dias_desde_ultima_publicacao_{sufixo}")
        ])
        .drop([
            f"data_inicio_janela_{sufixo}",
            f"ultima_publicacao_{sufixo}",
            f"qtd_dias_janela_{sufixo}"
        ])
    )

    return covs

In [ ]:
covs_1m = criar_covariaveis_janela(
    matriz_mod=base_modelagem,
    df_noticias_com_id=df_noticias_com_id,
    janela_meses=1
)

covs_3m = criar_covariaveis_janela(
    matriz_mod=base_modelagem,
    df_noticias_com_id=df_noticias_com_id,
    janela_meses=3
)

covs_6m = criar_covariaveis_janela(
    matriz_mod=base_modelagem,
    df_noticias_com_id=df_noticias_com_id,
    janela_meses=6
)

In [ ]:
matriz_mod_cov = (
    base_modelagem
    .join(covs_1m, on=["id_autor", "data_ref"], how="left")
    .join(covs_3m, on=["id_autor", "data_ref"], how="left")
    .join(covs_6m, on=["id_autor", "data_ref"], how="left")
)

In [ ]:
matriz_mod_cov = matriz_mod_cov.with_columns([
    pl.when(pl.col("qtd_publicacoes_3m") > 0)
    .then(pl.col("qtd_publicacoes_1m") / pl.col("qtd_publicacoes_3m"))
    .otherwise(None)
    .alias("razao_qtd_1m_sobre_3m"),

    pl.when(pl.col("qtd_publicacoes_6m") > 0)
    .then(pl.col("qtd_publicacoes_3m") / pl.col("qtd_publicacoes_6m"))
    .otherwise(None)
    .alias("razao_qtd_3m_sobre_6m"),

    (
        pl.col("qtd_publicacoes_1m") -
        pl.col("media_publicacoes_mes_3m")
    )
    .alias("dif_qtd_1m_vs_media_3m"),

    (
        pl.col("media_publicacoes_mes_3m") -
        pl.col("media_publicacoes_mes_6m")
    )
    .alias("dif_media_3m_vs_media_6m"),

    pl.when(pl.col("media_publicacoes_mes_6m") > 0)
    .then(
        pl.col("media_publicacoes_mes_3m") /
        pl.col("media_publicacoes_mes_6m")
    )
    .otherwise(None)
    .alias("razao_media_3m_sobre_6m")
])

In [ ]:
cols_1m = [
    "qtd_publicacoes_1m",
    "qtd_dias_publicados_1m",
    "qtd_meses_com_publicacao_1m",
    "flag_publicou_1m",
    "media_publicacoes_mes_1m",
    "prop_meses_com_publicacao_1m",
    "prop_dias_com_publicacao_1m",
    "dias_desde_ultima_publicacao_1m",
    "tam_medio_texto_1m",
    "tam_total_texto_1m",
    "tam_medio_titulo_1m",
    "qtd_paginas_origem_distintas_1m"
]

cols_3m = [
    "qtd_publicacoes_3m",
    "qtd_dias_publicados_3m",
    "qtd_meses_com_publicacao_3m",
    "flag_publicou_3m",
    "media_publicacoes_mes_3m",
    "prop_meses_com_publicacao_3m",
    "prop_dias_com_publicacao_3m",
    "dias_desde_ultima_publicacao_3m",
    "tam_medio_texto_3m",
    "tam_total_texto_3m",
    "tam_medio_titulo_3m",
    "qtd_paginas_origem_distintas_3m"
]


cols_6m = [
    "qtd_publicacoes_6m",
    "qtd_dias_publicados_6m",
    "qtd_meses_com_publicacao_6m",
    "flag_publicou_6m",
    "media_publicacoes_mes_6m",
    "prop_meses_com_publicacao_6m",
    "prop_dias_com_publicacao_6m",
    "dias_desde_ultima_publicacao_6m",
    "tam_medio_texto_6m",
    "tam_total_texto_6m",
    "tam_medio_titulo_6m",
    "qtd_paginas_origem_distintas_6m"
]

In [ ]:
# Primeiro mascara as covariáveis por relacionamento
matriz_mod_cov = (
    matriz_mod_cov
    .with_columns([
        pl.when(pl.col("flag_1m_relacionamento") == 1)
        .then(pl.col(c))
        .otherwise(None)
        .alias(c)
        for c in cols_1m
    ])
    .with_columns([
        pl.when(pl.col("flag_3m_relacionamento") == 1)
        .then(pl.col(c))
        .otherwise(None)
        .alias(c)
        for c in cols_3m
    ])
    .with_columns([
        pl.when(pl.col("flag_6m_relacionamento") == 1)
        .then(pl.col(c))
        .otherwise(None)
        .alias(c)
        for c in cols_6m
    ])
)

# Depois cria variáveis comparativas/tendência
matriz_mod_cov = matriz_mod_cov.with_columns([
    pl.when(
        (pl.col("qtd_publicacoes_3m").is_not_null()) &
        (pl.col("qtd_publicacoes_3m") > 0)
    )
    .then(pl.col("qtd_publicacoes_1m") / pl.col("qtd_publicacoes_3m"))
    .otherwise(None)
    .alias("razao_qtd_1m_sobre_3m"),

    pl.when(
        (pl.col("qtd_publicacoes_6m").is_not_null()) &
        (pl.col("qtd_publicacoes_6m") > 0)
    )
    .then(pl.col("qtd_publicacoes_3m") / pl.col("qtd_publicacoes_6m"))
    .otherwise(None)
    .alias("razao_qtd_3m_sobre_6m"),

    pl.when(
        (pl.col("media_publicacoes_mes_3m").is_not_null()) &
        (pl.col("qtd_publicacoes_1m").is_not_null())
    )
    .then(pl.col("qtd_publicacoes_1m") - pl.col("media_publicacoes_mes_3m"))
    .otherwise(None)
    .alias("dif_qtd_1m_vs_media_3m"),

    pl.when(
        (pl.col("media_publicacoes_mes_3m").is_not_null()) &
        (pl.col("media_publicacoes_mes_6m").is_not_null())
    )
    .then(pl.col("media_publicacoes_mes_3m") - pl.col("media_publicacoes_mes_6m"))
    .otherwise(None)
    .alias("dif_media_3m_vs_media_6m")
])

# Mais covariaveis

In [ ]:
import polars as pl

def to_lazy(df):
    return df.lazy() if isinstance(df, pl.DataFrame) else df

lf_base = matriz_mod_cov
lf_noticias = df_noticias_com_id

ultima_pub_ate_ref = (
    lf_base
    .select(["id_autor", "data_ref"])
    .unique()
    .join(
        lf_noticias
        .select(["id_autor", "data_iso"])
        .filter(
            pl.col("id_autor").is_not_null() &
            pl.col("data_iso").is_not_null()
        ),
        on="id_autor",
        how="left"
    )
    .filter(
        pl.col("data_iso") < pl.col("data_ref")
    )
    .group_by(["id_autor", "data_ref"])
    .agg(
        pl.col("data_iso").max().alias("ultima_publicacao_ate_ref")
    )
)

matriz_mod_cov = (
    lf_base
    .join(
        ultima_pub_ate_ref,
        on=["id_autor", "data_ref"],
        how="left"
    )
    .with_columns(
        pl.when(pl.col("ultima_publicacao_ate_ref").is_not_null())
        .then(
            (pl.col("data_ref") - pl.col("ultima_publicacao_ate_ref"))
            .dt.total_days()
        )
        .otherwise(None)
        .alias("dias_desde_ultima_publicacao_ate_ref")
    )
    .drop("ultima_publicacao_ate_ref")
)

matriz_mod_cov = matriz_mod_cov.with_columns([
    pl.when(pl.col("dias_desde_ultima_publicacao_ate_ref") >= 30)
    .then(1)
    .otherwise(0)
    .alias("flag_sem_publicar_30d"),

    pl.when(pl.col("dias_desde_ultima_publicacao_ate_ref") >= 45)
    .then(1)
    .otherwise(0)
    .alias("flag_sem_publicar_45d"),

    pl.when(pl.col("dias_desde_ultima_publicacao_ate_ref") >= 60)
    .then(1)
    .otherwise(0)
    .alias("flag_sem_publicar_60d"),
])

In [ ]:
matriz_mod_cov = matriz_mod_cov.with_columns([
    pl.when(
        (pl.col("qtd_publicacoes_1m").is_not_null()) &
        (pl.col("media_publicacoes_mes_3m").is_not_null()) &
        (pl.col("media_publicacoes_mes_3m") > 0)
    )
    .then(
        pl.col("qtd_publicacoes_1m") /
        pl.col("media_publicacoes_mes_3m")
    )
    .otherwise(None)
    .alias("indice_atividade_1m_vs_media_3m"),

    pl.when(
        (pl.col("media_publicacoes_mes_3m").is_not_null()) &
        (pl.col("media_publicacoes_mes_6m").is_not_null()) &
        (pl.col("media_publicacoes_mes_6m") > 0)
    )
    .then(
        pl.col("media_publicacoes_mes_3m") /
        pl.col("media_publicacoes_mes_6m")
    )
    .otherwise(None)
    .alias("indice_atividade_3m_vs_media_6m"),

    (
        pl.col("media_publicacoes_mes_3m") -
        pl.col("media_publicacoes_mes_6m")
    )
    .alias("delta_media_publicacoes_3m_6m")
])

In [ ]:
def criar_qtd_publicacoes_janela(matriz, noticias, inicio_meses_atras, fim_meses_atras, prefixo):
    lf_matriz = matriz
    lf_noticias = noticias

    base_ref = (
        lf_matriz
        .select(["id_autor", "data_ref"])
        .unique()
        .with_columns([
            pl.col("data_ref")
            .dt.offset_by(f"-{inicio_meses_atras}mo")
            .alias("data_inicio_janela"),

            pl.col("data_ref")
            .dt.offset_by(f"-{fim_meses_atras}mo")
            .alias("data_fim_janela")
        ])
    )

    cov = (
        base_ref
        .join(
            lf_noticias
            .select(["id_autor", "data_iso"])
            .filter(
                pl.col("id_autor").is_not_null() &
                pl.col("data_iso").is_not_null()
            ),
            on="id_autor",
            how="left"
        )
        .filter(
            (pl.col("data_iso") >= pl.col("data_inicio_janela")) &
            (pl.col("data_iso") < pl.col("data_fim_janela"))
        )
        .group_by(["id_autor", "data_ref"])
        .agg(
            pl.len().alias(f"qtd_publicacoes_{prefixo}")
        )
    )

    cov = (
        base_ref
        .select(["id_autor", "data_ref"])
        .join(cov, on=["id_autor", "data_ref"], how="left")
        .with_columns(
            pl.col(f"qtd_publicacoes_{prefixo}").fill_null(0)
        )
    )

    return cov


qtd_prev_1m = criar_qtd_publicacoes_janela(
    matriz=matriz_mod_cov,
    noticias=df_noticias_com_id,
    inicio_meses_atras=2,
    fim_meses_atras=1,
    prefixo="prev_1m"
)

qtd_prev_3m = criar_qtd_publicacoes_janela(
    matriz=matriz_mod_cov,
    noticias=df_noticias_com_id,
    inicio_meses_atras=6,
    fim_meses_atras=3,
    prefixo="prev_3m"
)


matriz_mod_cov = (
    matriz_mod_cov
    .join(qtd_prev_1m, on=["id_autor", "data_ref"], how="left")
    .join(qtd_prev_3m, on=["id_autor", "data_ref"], how="left")
)


matriz_mod_cov = matriz_mod_cov.with_columns([
    (
        pl.col("qtd_publicacoes_1m") -
        pl.col("qtd_publicacoes_prev_1m")
    )
    .alias("delta_qtd_1m_vs_prev_1m"),

    pl.when(pl.col("qtd_publicacoes_prev_1m") > 0)
    .then(
        pl.col("qtd_publicacoes_1m") /
        pl.col("qtd_publicacoes_prev_1m")
    )
    .otherwise(None)
    .alias("razao_qtd_1m_vs_prev_1m"),

    (
        pl.col("qtd_publicacoes_3m") -
        pl.col("qtd_publicacoes_prev_3m")
    )
    .alias("delta_qtd_3m_vs_prev_3m"),

    pl.when(pl.col("qtd_publicacoes_prev_3m") > 0)
    .then(
        pl.col("qtd_publicacoes_3m") /
        pl.col("qtd_publicacoes_prev_3m")
    )
    .otherwise(None)
    .alias("razao_qtd_3m_vs_prev_3m")
])

In [ ]:
lf_base = matriz_mod_cov
lf_noticias = df_noticias_com_id

base_ref_6m = (
    lf_base
    .select(["id_autor", "data_ref"])
    .unique()
    .with_columns(
        pl.col("data_ref")
        .dt.offset_by("-6mo")
        .alias("data_inicio_6m")
    )
)

dias_publicados_6m = (
    base_ref_6m
    .join(
        lf_noticias
        .select(["id_autor", "data_iso"])
        .filter(
            pl.col("id_autor").is_not_null() &
            pl.col("data_iso").is_not_null()
        ),
        on="id_autor",
        how="left"
    )
    .filter(
        (pl.col("data_iso") >= pl.col("data_inicio_6m")) &
        (pl.col("data_iso") < pl.col("data_ref"))
    )
    .select(["id_autor", "data_ref", "data_iso"])
    .unique()
    .sort(["id_autor", "data_ref", "data_iso"])
    .with_columns(
        (
            pl.col("data_iso") -
            pl.col("data_iso").shift(1).over(["id_autor", "data_ref"])
        )
        .dt.total_days()
        .alias("gap_dias")
    )
)

cov_gap_6m = (
    dias_publicados_6m
    .group_by(["id_autor", "data_ref"])
    .agg([
        pl.col("gap_dias").mean().alias("media_gap_dias_6m"),
        pl.col("gap_dias").max().alias("max_gap_dias_6m"),
        pl.col("gap_dias").std().alias("std_gap_dias_6m")
    ])
)

matriz_mod_cov = (
    matriz_mod_cov
    .join(
        cov_gap_6m,
        on=["id_autor", "data_ref"],
        how="left"
    )
)

In [ ]:
matriz_mod_cov = matriz_mod_cov.with_columns([
    pl.when(
        (pl.col("qtd_publicacoes_1m") == 0) &
        (pl.col("qtd_publicacoes_3m") > 0)
    )
    .then(1)
    .otherwise(0)
    .alias("flag_zerou_1m_mas_publicou_3m"),

    pl.when(
        (pl.col("qtd_publicacoes_1m") == 0) &
        (pl.col("qtd_publicacoes_6m") > 0)
    )
    .then(1)
    .otherwise(0)
    .alias("flag_zerou_1m_mas_publicou_6m"),

    pl.when(
        (pl.col("qtd_publicacoes_3m") == 0) &
        (pl.col("qtd_publicacoes_6m") > 0)
    )
    .then(1)
    .otherwise(0)
    .alias("flag_zerou_3m_mas_publicou_6m"),

    pl.when(
        pl.col("indice_atividade_1m_vs_media_3m") < 0.5
    )
    .then(1)
    .otherwise(0)
    .alias("flag_queda_forte_atividade_1m"),

    pl.when(
        pl.col("indice_atividade_3m_vs_media_6m") < 0.5
    )
    .then(1)
    .otherwise(0)
    .alias("flag_queda_forte_atividade_3m")
])

In [ ]:
matriz_mod_cov = matriz_mod_cov.with_columns([
    
    # ------------------------------------------------------------
    # Recência geral até a data_ref
    # Só faz sentido se o autor tiver pelo menos 1 mês de relacionamento
    # ------------------------------------------------------------
    pl.when(
        pl.col("flag_1m_relacionamento") == 1
    )
    .then(pl.col("dias_desde_ultima_publicacao_ate_ref"))
    .otherwise(None)
    .alias("dias_desde_ultima_publicacao_ate_ref"),

    pl.when(
        (pl.col("flag_1m_relacionamento") == 1) &
        (pl.col("dias_desde_ultima_publicacao_ate_ref") >= 30)
    )
    .then(1)
    .when(pl.col("flag_1m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_sem_publicar_30d"),

    pl.when(
        (pl.col("flag_3m_relacionamento") == 1) &
        (pl.col("dias_desde_ultima_publicacao_ate_ref") >= 45)
    )
    .then(1)
    .when(pl.col("flag_3m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_sem_publicar_45d"),

    pl.when(
        (pl.col("flag_3m_relacionamento") == 1) &
        (pl.col("dias_desde_ultima_publicacao_ate_ref") >= 60)
    )
    .then(1)
    .when(pl.col("flag_3m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_sem_publicar_60d"),
])


matriz_mod_cov = matriz_mod_cov.with_columns([

    # ------------------------------------------------------------
    # Último 1m contra média dos últimos 3m
    # Precisa de relacionamento de 3 meses
    # ------------------------------------------------------------
    pl.when(
        (pl.col("flag_3m_relacionamento") == 1) &
        (pl.col("media_publicacoes_mes_3m").is_not_null()) &
        (pl.col("media_publicacoes_mes_3m") > 0)
    )
    .then(
        pl.col("qtd_publicacoes_1m") /
        pl.col("media_publicacoes_mes_3m")
    )
    .otherwise(None)
    .alias("indice_atividade_1m_vs_media_3m"),

    pl.when(
        (pl.col("flag_3m_relacionamento") == 1) &
        (pl.col("qtd_publicacoes_1m").is_not_null()) &
        (pl.col("media_publicacoes_mes_3m").is_not_null())
    )
    .then(
        pl.col("qtd_publicacoes_1m") -
        pl.col("media_publicacoes_mes_3m")
    )
    .otherwise(None)
    .alias("delta_atividade_1m_vs_media_3m"),

    # ------------------------------------------------------------
    # Últimos 3m contra média dos últimos 6m
    # Precisa de relacionamento de 6 meses
    # ------------------------------------------------------------
    pl.when(
        (pl.col("flag_6m_relacionamento") == 1) &
        (pl.col("media_publicacoes_mes_6m").is_not_null()) &
        (pl.col("media_publicacoes_mes_6m") > 0)
    )
    .then(
        pl.col("media_publicacoes_mes_3m") /
        pl.col("media_publicacoes_mes_6m")
    )
    .otherwise(None)
    .alias("indice_atividade_3m_vs_media_6m"),

    pl.when(
        (pl.col("flag_6m_relacionamento") == 1) &
        (pl.col("media_publicacoes_mes_3m").is_not_null()) &
        (pl.col("media_publicacoes_mes_6m").is_not_null())
    )
    .then(
        pl.col("media_publicacoes_mes_3m") -
        pl.col("media_publicacoes_mes_6m")
    )
    .otherwise(None)
    .alias("delta_media_publicacoes_3m_6m"),
])


matriz_mod_cov = matriz_mod_cov.with_columns([

    # Precisa de 3 meses, pois usa qtd_publicacoes_1m e qtd_publicacoes_3m
    pl.when(
        (pl.col("flag_3m_relacionamento") == 1) &
        (pl.col("qtd_publicacoes_1m") == 0) &
        (pl.col("qtd_publicacoes_3m") > 0)
    )
    .then(1)
    .when(pl.col("flag_3m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_zerou_1m_mas_publicou_3m"),

    # Precisa de 6 meses
    pl.when(
        (pl.col("flag_6m_relacionamento") == 1) &
        (pl.col("qtd_publicacoes_1m") == 0) &
        (pl.col("qtd_publicacoes_6m") > 0)
    )
    .then(1)
    .when(pl.col("flag_6m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_zerou_1m_mas_publicou_6m"),

    # Precisa de 6 meses
    pl.when(
        (pl.col("flag_6m_relacionamento") == 1) &
        (pl.col("qtd_publicacoes_3m") == 0) &
        (pl.col("qtd_publicacoes_6m") > 0)
    )
    .then(1)
    .when(pl.col("flag_6m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_zerou_3m_mas_publicou_6m"),

    # Precisa de 3 meses
    pl.when(
        (pl.col("flag_3m_relacionamento") == 1) &
        (pl.col("indice_atividade_1m_vs_media_3m") < 0.5)
    )
    .then(1)
    .when(pl.col("flag_3m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_queda_forte_atividade_1m"),

    # Precisa de 6 meses
    pl.when(
        (pl.col("flag_6m_relacionamento") == 1) &
        (pl.col("indice_atividade_3m_vs_media_6m") < 0.5)
    )
    .then(1)
    .when(pl.col("flag_6m_relacionamento") == 1)
    .then(0)
    .otherwise(None)
    .alias("flag_queda_forte_atividade_3m"),
])

In [ ]:
cols_1m = [
    "qtd_publicacoes_1m",
    "qtd_dias_publicados_1m",
    "qtd_meses_com_publicacao_1m",
    "tam_medio_texto_1m",
    "tam_total_texto_1m",
    "tam_medio_titulo_1m",
    "qtd_paginas_origem_distintas_1m",
    "flag_publicou_1m",
    "media_publicacoes_mes_1m",
    "prop_meses_com_publicacao_1m",
    "prop_dias_com_publicacao_1m",
    "dias_desde_ultima_publicacao_1m",
]

cols_3m = [
    "qtd_publicacoes_3m",
    "qtd_dias_publicados_3m",
    "qtd_meses_com_publicacao_3m",
    "tam_medio_texto_3m",
    "tam_total_texto_3m",
    "tam_medio_titulo_3m",
    "qtd_paginas_origem_distintas_3m",
    "flag_publicou_3m",
    "media_publicacoes_mes_3m",
    "prop_meses_com_publicacao_3m",
    "prop_dias_com_publicacao_3m",
    "dias_desde_ultima_publicacao_3m",

    # novas que dependem de 3m
    "flag_sem_publicar_45d",
    "flag_sem_publicar_60d",
    "indice_atividade_1m_vs_media_3m",
    "delta_atividade_1m_vs_media_3m",
    "flag_zerou_1m_mas_publicou_3m",
    "flag_queda_forte_atividade_1m",
]

cols_6m = [
    "qtd_publicacoes_6m",
    "qtd_dias_publicados_6m",
    "qtd_meses_com_publicacao_6m",
    "tam_medio_texto_6m",
    "tam_total_texto_6m",
    "tam_medio_titulo_6m",
    "qtd_paginas_origem_distintas_6m",
    "flag_publicou_6m",
    "media_publicacoes_mes_6m",
    "prop_meses_com_publicacao_6m",
    "prop_dias_com_publicacao_6m",
    "dias_desde_ultima_publicacao_6m",

    # novas que dependem de 6m
    "indice_atividade_3m_vs_media_6m",
    "delta_media_publicacoes_3m_6m",
    "flag_zerou_1m_mas_publicou_6m",
    "flag_zerou_3m_mas_publicou_6m",
    "flag_queda_forte_atividade_3m",

    # se você criou gaps de 6m
    "media_gap_dias_6m",
    "max_gap_dias_6m",
    "std_gap_dias_6m",
]


def aplicar_mascara_relacionamento(df, cols, flag):
    schema = df.collect_schema() if isinstance(df, pl.LazyFrame) else df.schema

    cols_existentes = [
        c for c in cols
        if c in schema
    ]

    return df.with_columns([
        pl.when(pl.col(flag) == 1)
        .then(pl.col(c))
        .otherwise(None)
        .alias(c)
        for c in cols_existentes
    ])


matriz_mod_cov = aplicar_mascara_relacionamento(
    matriz_mod_cov,
    cols_1m,
    "flag_1m_relacionamento"
)

matriz_mod_cov = aplicar_mascara_relacionamento(
    matriz_mod_cov,
    cols_3m,
    "flag_3m_relacionamento"
)

matriz_mod_cov = aplicar_mascara_relacionamento(
    matriz_mod_cov,
    cols_6m,
    "flag_6m_relacionamento"
)

# Aplicar floresta aleatoria


# Bagging

In [ ]:
df_modelo = (
    matriz_mod_cov.collect()
    if isinstance(matriz_mod_cov, pl.LazyFrame)
    else matriz_mod_cov
)

df_modelo = (
    df_modelo
    .filter(
        (pl.col("SEGMENTO") == "MATURADO") &
        (pl.col("conceito_demissao_3m_real").is_not_null())
    )
)

target = "conceito_demissao_3m_real"

colunas_remover = [
    # resposta e derivadas da resposta
    "conceito_demissao_3m_real",

    # identificadores e datas
    "id_autor",
    "autor",
    "data_ref",
    "menor_data_publicacao",

    # segmentação e flags que você não quer usar
    "SEGMENTO",
    "flag_1m_relacionamento",
    "flag_3m_relacionamento",
    "flag_6m_relacionamento",

    # amostras, mesmo que você não vá usar
    "AMOSTRA",
]

tipos_numericos = [
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64
]

features = [
    col for col, dtype in df_modelo.schema.items()
    if col not in colunas_remover
    and dtype in tipos_numericos
]


features

In [ ]:
X = df_modelo.select(features).to_pandas()
y = df_modelo.select(target).to_pandas()[target].astype(int)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


rf_oob = Pipeline(steps=[
    ("model", RandomForestClassifier(
        n_estimators=10000,
        max_features="sqrt",
        bootstrap=True,
        oob_score=True,
        class_weight=None,
        random_state=42,
        n_jobs=-1
    ))
])

rf_oob.fit(X, y)

In [ ]:
modelo_rf = rf_oob.named_steps["model"]

print(f"Acurácia OOB: {modelo_rf.oob_score_:.4f}")
print(f"Erro OOB: {1 - modelo_rf.oob_score_:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

prob_oob = modelo_rf.oob_decision_function_[:, 1]

y_pred_oob = (prob_oob >= 0.5).astype(int)

acc_oob = accuracy_score(y, y_pred_oob)
erro_oob = 1 - acc_oob
auc_oob = roc_auc_score(y, prob_oob)
precision_oob = precision_score(y, y_pred_oob, zero_division=0)
recall_oob = recall_score(y, y_pred_oob, zero_division=0)
f1_oob = f1_score(y, y_pred_oob, zero_division=0)

print(f"Acurácia OOB: {acc_oob:.4f}")
print(f"Erro OOB: {erro_oob:.4f}")
print(f"AUC OOB: {auc_oob:.4f}")
print(f"Precision OOB: {precision_oob:.4f}")
print(f"Recall OOB: {recall_oob:.4f}")
print(f"F1-score OOB: {f1_oob:.4f}")

print("\nMatriz de confusão OOB:")
print(confusion_matrix(y, y_pred_oob))

print("\nRelatório OOB:")
print(classification_report(y, y_pred_oob))

In [ ]:
import pandas as pd
metricas_oob = pd.DataFrame({
    "metrica": [
        "Acurácia OOB",
        "Erro OOB",
        "AUC OOB",
        "Precision OOB",
        "Recall OOB",
        "F1-score OOB"
    ],
    "valor": [
        acc_oob,
        erro_oob,
        auc_oob,
        precision_oob,
        recall_oob,
        f1_oob
    ]
})

metricas_oob

In [ ]:
importancia = (
    pd.DataFrame({
        "variavel": features,
        "importancia": modelo_rf.feature_importances_
    })
    .sort_values("importancia", ascending=False)
)

importancia.head(30)